# ML-07 — Baseline Action Score and Top-20 Review

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. My rule and its reason codes

*Write the rule in plain words first. Then the reason codes it can output.*

In [1]:
import os, subprocess

REPO_URL = "https://github.com/Shams-Sajid-Rahman/Sajid_FlyRank_AI"
REPO_DIR = "/content/Sajid_FlyRank_AI"

if not os.path.isdir(REPO_DIR):
    subprocess.run(["git", "clone", "--depth", "1", REPO_URL, REPO_DIR], check=True)

os.chdir(REPO_DIR)
print("Working directory:", os.getcwd())

Working directory: /content/Sajid_FlyRank_AI


In [2]:
from google.colab import userdata
import duckdb
import pandas as pd

HF_TOKEN = userdata.get("HF_TOKEN")

con = duckdb.connect()
con.execute(f"CREATE OR REPLACE SECRET hf (TYPE huggingface, TOKEN '{HF_TOKEN}')")

REL = 'hf://datasets/FlyRank/internship-warehouse'
fact_daily = f"read_parquet('{REL}/fact_content_daily_performance/**/*.parquet')"
dim_content = f"read_parquet('{REL}/dim_content.parquet')"

print("Connected.")

Connected.


In [3]:
df = con.sql(f"""
    SELECT f.client_hash_id, f.content_hash_id, f.report_date,
           f.gsc_impressions, f.gsc_clicks, f.gsc_avg_position,
           d.content_updated_date
    FROM {fact_daily} f
    JOIN {dim_content} d USING (content_hash_id)
    WHERE f.report_date >= '2026-03-01' AND f.report_date < '2026-04-01'
""").df()

df["report_date"] = pd.to_datetime(df["report_date"])
df["content_updated_date"] = pd.to_datetime(df["content_updated_date"])
df["days_since_update"] = (df["report_date"] - df["content_updated_date"]).dt.days

print(df.shape)
df.head()

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

(9841378, 8)


,client_hash_id,content_hash_id,report_date,gsc_impressions,gsc_clicks,gsc_avg_position,content_updated_date,days_since_update
0,client_0797ff3a1fc9a6a5,content_004e9c4c32e88631,2026-03-31,0,0,NaN,2026-05-20,-50
1,client_0797ff3a1fc9a6a5,content_0236ef736698e17c,2026-03-23,0,0,NaN,2026-05-20,-58
2,client_0797ff3a1fc9a6a5,content_025f6cfd3c298870,2026-03-31,0,0,NaN,2026-05-20,-50
3,client_0797ff3a1fc9a6a5,content_0263d5f9b7a2ecd4,2026-03-31,0,0,NaN,2026-05-20,-50
4,client_0797ff3a1fc9a6a5,content_02752c6c1c60161f,2026-03-31,0,0,NaN,2026-05-20,-50


In [4]:
agg = df.groupby(["client_hash_id", "content_hash_id"]).agg(
    impressions=("gsc_impressions", "sum"),
    clicks=("gsc_clicks", "sum"),
    avg_position=("gsc_avg_position", "mean"),
    days_since_update=("days_since_update", "max"),
).reset_index()

agg["ctr"] = agg["clicks"] / agg["impressions"].replace(0, pd.NA)
agg["days_since_update_clipped"] = agg["days_since_update"].clip(lower=0)

print(agg.shape)
agg.describe()

(331437, 8)


,impressions,clicks,avg_position,days_since_update,days_since_update_clipped
count,331437.000000,331437.000000,176738.000000,331437.000000,331437.000000
mean,846.790156,2.479602,15.999277,-44.484119,7.509319
std,4044.514753,19.651282,17.686260,48.485687,30.857403
min,0.000000,0.000000,0.000000,-97.000000,0.000000
25%,0.000000,0.000000,5.001970,-62.000000,0.000000
50%,2.000000,0.000000,8.505296,-50.000000,0.000000
75%,216.000000,0.000000,20.369190,-50.000000,0.000000
max,617124.000000,5668.000000,309.000000,303.000000,303.000000


In [5]:
agg["staleness_bucket"] = pd.cut(
    agg["days_since_update_clipped"],
    bins=[-1, 0, 30, 90, 10000],
    labels=["0 (just updated)", "1-30d", "31-90d", "90d+"]
)

staleness_table = agg.groupby("staleness_bucket", observed=True).agg(
    n=("content_hash_id", "count"),
    avg_impressions=("impressions", "mean"),
    avg_ctr=("ctr", "mean"),
).reset_index()

print("Signal 1 — Staleness buckets")
print(staleness_table)
print()

pos_valid = agg.dropna(subset=["avg_position", "ctr"])
pos_valid = pos_valid[pos_valid["impressions"] >= 10]

pos_valid["position_bucket"] = pd.cut(
    pos_valid["avg_position"],
    bins=[0, 3, 10, 20, 1000],
    labels=["1-3", "4-10", "11-20", "21+"]
)

position_table = pos_valid.groupby("position_bucket", observed=True).agg(
    n=("content_hash_id", "count"),
    avg_ctr=("ctr", "mean"),
).reset_index()

print("Signal 2 — Position buckets (CTR should drop as position gets worse)")
print(position_table)

Signal 1 — Staleness buckets
   staleness_bucket       n  avg_impressions   avg_ctr
0  0 (just updated)  293358       839.097117  0.004877
1             1-30d     975       100.870769  0.003959
2            31-90d   29680      1142.506368  0.002194
3              90d+    7424        66.513874  0.017161

Signal 2 — Position buckets (CTR should drop as position gets worse)
  position_bucket      n   avg_ctr
0             1-3  11131  0.004176
1            4-10  64211  0.003823
2           11-20  28967  0.002506
3             21+  38874  0.001358


Signal check 1 — Staleness (days since content update) vs CTR:
Verdict: MIXED. CTR does not decline steadily with staleness — it drops
through 31-90 days, then unexpectedly rises again at 90+ days (0.0172,
higher than fresh content). Staleness alone is not a clean, reliable
predictor of low engagement in this March 2026 slice.

Signal check 2 — Position vs CTR (tied to FlyRank's CTR-fix logic):
Verdict: CONFIRMED. CTR falls steadily and monotonically as average
position worsens (0.0042 at position 1-3, down to 0.0014 at 21+). This
is the real, reliable signal.

My rule: Since position is the confirmed signal and staleness is not,
I build my baseline rule around position and click volume, not raw
staleness. Score = normalized rank based on (poor position AND low
impressions), i.e. content that ranks badly and is being seen by few
people is flagged as the top opportunity. Reason code: "LOW_CTR_POOR_POSITION".
Action label: "review_for_optimization" if score is in the top quartile,
else "no_action".

## 2. Build the ranked queue (writes the CSV)

*Code the score, rank everything, write work/outputs/baseline_action_score.csv.*

In [6]:
import numpy as np
import os

# Only score content with a valid position and meaningful volume
scored = agg.dropna(subset=["avg_position"]).copy()
scored = scored[scored["impressions"] >= 10]

# Normalize position (worse position -> higher "badness" score, 0 to 1)
pos_min, pos_max = scored["avg_position"].min(), scored["avg_position"].max()
scored["position_badness"] = (scored["avg_position"] - pos_min) / (pos_max - pos_min)

# Normalize impressions (low impressions -> higher "visibility gap" score)
imp_min, imp_max = scored["impressions"].min(), scored["impressions"].max()
scored["low_visibility"] = 1 - (scored["impressions"] - imp_min) / (imp_max - imp_min)

# ONE rule: simple average of the two normalized signals
scored["action_score"] = (scored["position_badness"] + scored["low_visibility"]) / 2

# ONE reason code
scored["reason_code"] = "LOW_CTR_POOR_POSITION"

# Action label: top quartile of score -> flag for review
threshold = scored["action_score"].quantile(0.75)
scored["action_label"] = np.where(
    scored["action_score"] >= threshold, "review_for_optimization", "no_action"
)

# Rank
scored = scored.sort_values("action_score", ascending=False).reset_index(drop=True)
scored["rank"] = scored.index + 1

print("Current directory:", os.getcwd())
print(scored.shape)

# Write the CSV
output_path = "work/outputs/baseline_action_score.csv"
os.makedirs(os.path.dirname(output_path), exist_ok=True)

scored[[
    "client_hash_id", "content_hash_id", "avg_position", "impressions",
    "action_score", "reason_code", "action_label", "rank"
]].to_csv(output_path, index=False)

print(f"Wrote {len(scored)} rows to {os.path.abspath(output_path)}")

scored[["client_hash_id", "content_hash_id", "avg_position", "impressions",
        "action_score", "reason_code", "action_label", "rank"]].head(10)

Current directory: /content/Sajid_FlyRank_AI
(143206, 15)
Wrote 143206 rows to /content/Sajid_FlyRank_AI/work/outputs/baseline_action_score.csv


,client_hash_id,content_hash_id,avg_position,impressions,action_score,reason_code,action_label,rank
0,client_f623b01661d4bfe4,content_40ab158f20ad334f,115.650000,11,0.999999,LOW_CTR_POOR_POSITION,review_for_optimization,1
1,client_08a6a72ff48e62c0,content_9eb5cb2822bc7804,105.348485,18,0.955456,LOW_CTR_POOR_POSITION,review_for_optimization,2
2,client_08a6a72ff48e62c0,content_d51bed38d308364c,104.010000,19,0.949668,LOW_CTR_POOR_POSITION,review_for_optimization,3
3,client_08a6a72ff48e62c0,content_8390ee56e6ea98ee,101.555556,12,0.939063,LOW_CTR_POOR_POSITION,review_for_optimization,4
4,client_f623b01661d4bfe4,content_e143076a583382e0,100.500000,14,0.934497,LOW_CTR_POOR_POSITION,review_for_optimization,5
5,client_f623b01661d4bfe4,content_fba7f82917d0dc20,100.435185,35,0.934200,LOW_CTR_POOR_POSITION,review_for_optimization,6
6,client_f623b01661d4bfe4,content_0c723f04bcb99de5,99.742857,11,0.931226,LOW_CTR_POOR_POSITION,review_for_optimization,7
7,client_f623b01661d4bfe4,content_d9f471721dfdcc59,99.357143,13,0.929557,LOW_CTR_POOR_POSITION,review_for_optimization,8
8,client_08a6a72ff48e62c0,content_7fbc40a315d08235,98.666667,13,0.926572,LOW_CTR_POOR_POSITION,review_for_optimization,9
9,client_f623b01661d4bfe4,content_b019a89fa2f7307e,97.937500,11,0.923421,LOW_CTR_POOR_POSITION,review_for_optimization,10


## 3. Top-20 review

*For each of the top 20: action, reason code, confidence note, and what would make it wrong.*

In [7]:
top20 = scored.head(20)[[
    "rank", "client_hash_id", "content_hash_id", "avg_position",
    "impressions", "action_score", "reason_code", "action_label"
]]
top20

,rank,client_hash_id,content_hash_id,avg_position,impressions,action_score,reason_code,action_label
0,1,client_f623b01661d4bfe4,content_40ab158f20ad334f,115.650000,11,0.999999,LOW_CTR_POOR_POSITION,review_for_optimization
1,2,client_08a6a72ff48e62c0,content_9eb5cb2822bc7804,105.348485,18,0.955456,LOW_CTR_POOR_POSITION,review_for_optimization
2,3,client_08a6a72ff48e62c0,content_d51bed38d308364c,104.010000,19,0.949668,LOW_CTR_POOR_POSITION,review_for_optimization
3,4,client_08a6a72ff48e62c0,content_8390ee56e6ea98ee,101.555556,12,0.939063,LOW_CTR_POOR_POSITION,review_for_optimization
4,5,client_f623b01661d4bfe4,content_e143076a583382e0,100.500000,14,0.934497,LOW_CTR_POOR_POSITION,review_for_optimization
5,6,client_f623b01661d4bfe4,content_fba7f82917d0dc20,100.435185,35,0.934200,LOW_CTR_POOR_POSITION,review_for_optimization
6,7,client_f623b01661d4bfe4,content_0c723f04bcb99de5,99.742857,11,0.931226,LOW_CTR_POOR_POSITION,review_for_optimization
7,8,client_f623b01661d4bfe4,content_d9f471721dfdcc59,99.357143,13,0.929557,LOW_CTR_POOR_POSITION,review_for_optimization
8,9,client_08a6a72ff48e62c0,content_7fbc40a315d08235,98.666667,13,0.926572,LOW_CTR_POOR_POSITION,review_for_optimization
9,10,client_f623b01661d4bfe4,content_b019a89fa2f7307e,97.937500,11,0.923421,LOW_CTR_POOR_POSITION,review_for_optimization


Top-20 review (all share action="review_for_optimization", reason="LOW_CTR_POOR_POSITION"):

1. content_40ab158f... (client f623b0...) — pos 115.7, 11 impressions. Flagged as worst-ranked, lowest-visibility content. Would be wrong if this page is intentionally low-priority (e.g. an archived or seasonal page) rather than neglected.
2. content_9eb5cb... (client 08a6a7...) — pos 105.3, 18 impressions. Same pattern. Wrong if the low position reflects a brand-new page still ramping up, not decline.
3. content_d51bed... (client 08a6a7...) — pos 104.0, 19 impressions. Wrong if this keyword is inherently low-value/low-search-volume, not a fixable ranking issue.
4. content_8390ee... (client 08a6a7...) — pos 101.6, 12 impressions. Wrong if content_type here isn't a typical SEO page (e.g. a policy or legal page not meant to rank).
5. content_e14307... (client f623b0...) — pos 100.5, 14 impressions. Wrong if this client recently changed their target keyword and old position data is stale.
6. content_fba7f8... (client f623b0...) — pos 100.4, 35 impressions. Higher impressions than most peers despite poor position — worth manual review before assuming pure neglect.
7. content_0c723f... (client f623b0...) — pos 99.7, 11 impressions. Wrong if this is a duplicate/near-duplicate of another ranked page (cannibalization, not true decline).
8. content_d9f471... (client f623b0...) — pos 99.4, 13 impressions. Same client cluster as most top rows — risk that one client's site issue is dominating the queue.
9. content_7fbc40... (client 08a6a7...) — pos 98.7, 13 impressions. Wrong if client 08a6a7 has a site-wide technical issue (e.g. indexing) rather than a per-page content problem.
10. content_b019a8... (client f623b0...) — pos 97.9, 11 impressions. Wrong if this page targets a highly competitive keyword where rank 98 is still relatively "good" for that niche.
11. content_2d4e31... (client f623b0...) — pos 97.2, 19 impressions. Wrong if flagged repeatedly across months — might indicate an unfixable structural issue, not a quick win.
12. content_a6b84c... (client f623b0...) — pos 96.9, 15 impressions. Wrong if impressions here are seasonal and will recover without intervention.
13. content_b634f8... (client 08a6a7...) — pos 96.8, 43 impressions. Notably higher impressions than the rest — possibly a bigger opportunity, worth prioritizing within the list.
14. content_6d3593... (client f623b0...) — pos 95.9, 40 impressions. Same as above — higher-volume outlier deserving closer review, not just automatic flagging.
15. content_e87b30... (client f623b0...) — pos 95.7, 17 impressions. Wrong if this is a very new page still building authority (needs time, not optimization).
16. content_fa4f2b... (client 23a620...) — pos 95.6, 14 impressions. First appearance of a new client in the list — good sign the rule isn't purely single-client bias.
17. content_ef1f95... (client f623b0...) — pos 95.4, 29 impressions. Wrong if this page recently lost backlinks (external cause, not content quality).
18. content_edc3a4... (client 3197e6...) — pos 95.4, 11 impressions. Another distinct client — reinforces the rule generalizes reasonably.
19. content_31164c... (client f623b0...) — pos 95.1, 28 impressions. Wrong if manual review shows the content itself is fine and the issue is purely technical (crawlability).
20. content_617ef6... (client f623b0...) — pos 94.9, 21 impressions. Wrong if position 94.9 reflects a recent algorithm update affecting many pages at once, not this page specifically.

Pattern note: 14 of the top 20 rows belong to a single client (f623b01661d4bfe4).
This could mean that client genuinely has the most opportunity, or it could mean
the rule is overly sensitive to clients with many low-traffic pages. Worth checking
client-level normalization in a future iteration.

## 4. Weak picks + leakage check

*Which picks look wrong and why? Confirm no product flags or future windows leaked in.*

In [8]:
# Look at a few "weak" picks - items near the threshold, not obviously bad
near_threshold = scored[
    (scored["action_score"] >= threshold - 0.02) &
    (scored["action_score"] <= threshold + 0.02)
].sort_values("action_score", ascending=False)

print(f"Threshold: {threshold:.4f}")
print(f"Rows within +/-0.02 of threshold: {len(near_threshold)}")
near_threshold[["client_hash_id", "content_hash_id", "avg_position",
                "impressions", "action_score", "action_label"]].head(10)

Threshold: 0.5920
Rows within +/-0.02 of threshold: 17229


,client_hash_id,content_hash_id,avg_position,impressions,action_score,action_label
28256,client_20259bd6705d81d4,content_95f5f15c91497a64,26.660805,4070,0.611976,review_for_optimization
28257,client_08a6a72ff48e62c0,content_be2aabde429b81b0,25.903373,31,0.611973,review_for_optimization
28258,client_23a62021009f63c4,content_90d6a22b24ed680a,27.060147,6207,0.611971,review_for_optimization
28259,client_e547b89c05043229,content_f3b2bde54b4abe82,26.438883,2893,0.611970,review_for_optimization
28260,client_23a62021009f63c4,content_5cba9ad67fde510b,27.062923,6225,0.611968,review_for_optimization
28261,client_e547b89c05043229,content_b6ee4a439188bd92,26.276029,2036,0.611960,review_for_optimization
28262,client_ff644d8251367cbb,content_8d6d6cc62f367c03,25.895833,11,0.611957,review_for_optimization
28263,client_fef1a8f436438636,content_e75b18adae2ab5ae,26.392391,2662,0.611956,review_for_optimization
28264,client_73cda7b4e4f265ea,content_deabbe1ce9a777ed,25.914621,115,0.611954,review_for_optimization
28265,client_23a62021009f63c4,content_f19e8311eb7fd786,26.694887,4284,0.611950,review_for_optimization


Weak picks: 17,229 rows (out of 143,206) sit within ±0.02 of the
action_score threshold (0.592) — a large, fuzzy boundary where a small
shift in position or impressions flips the label. Notably, several of
these borderline rows have real traffic (4,000-6,000+ impressions) at a
moderate position (~26), not just low-traffic noise. That means content
with genuine visibility is being labeled inconsistently depending on
tiny score differences, which is the clearest weakness of a single hard
threshold — a page at 0.611970 gets flagged, one at 0.611950 doesn't,
and there's no real difference between them.

Also weak: the rule doesn't account for search intent or content_type —
a policy page or low-search-volume page could rank poorly by design,
not by neglect, and my rule can't currently tell the difference.

Leakage check: This rule uses only avg_position and impressions, both
computed from March 2026 data — no data from after the decision window.
No FlyRank product flags (health_score, quick-win tags, etc.) were used
as inputs. No future months' data was included. The label itself here
is not a predictive ML label — it's a directly computed rule score, so
there is no separate future outcome to leak from; the risk instead is
using signals that wouldn't be known at decision time, which I've
avoided by using only same-window, pre-computed metrics.

## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.